Even after manual edditing, there remained some sequences with high rates of clipping.
I create batch script to display them.

In [1]:
import pandas as pd
import re, io, subprocess
import os, sys

from Bio import SeqIO
from Bio.Seq import Seq
import pysam

In [2]:
file_list_path = '../data/File_list_20250204.csv'
clip_df_path = '../exp/dat/clipping_summary_cutadapt.csv'
is_pos_path = '../data/IS_positions.csv'
ref_fasta_dir = '../tmp/clean_fasta/'
bat_dir = '../exp/IGV_script/Clipping/'
bed_dir = '../exp/dat/bed/'
win_base_dir = r'V:\2022\analysis_2023\20250112_Revise\for_publication'
igv_snapshot_dir = win_base_dir + r'\exp\fig\IGV\Clipping'
igv_fasta_dir = win_base_dir + r'\tmp\clean_fasta'
igv_bam_dir = win_base_dir + r'\exp\bam\mm2_trimmed\all'
igv_bam_dir2 = win_base_dir + r'\exp\bam\mm2_cutadapt\all'
igv_bed_dir = win_base_dir + r'\exp\dat\bed'

os.makedirs(bat_dir, exist_ok=True)
os.makedirs(bed_dir, exist_ok=True)
igv_snapshot_dir_ = '../exp/fig/IGV/Clipping'
os.makedirs(igv_snapshot_dir_, exist_ok=True)

# trimmed (but not cut-adapted reads), all reads including those clipped
mm2_output_dir = os.path.join('../exp/bam/mm2_cutadapt/all/')
tmp_dir = '../tmp/fastq_Clipping'
dat_exp_dir = '../exp/dat'
fig_dir = '../exp/fig/clipping/cutadapt/igv'

os.makedirs(tmp_dir, exist_ok=True)
os.makedirs(dat_exp_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

In [3]:
ids = pd.read_csv(file_list_path)
ids.head(2)

,IS_Detect_ID,ParentLine,SubLine,gen,file_name,Anc,RecA,Prefix,sample_name_raw,Contig_Date,Complete,Folder,Folder_check,folder_err,File
0,NaN,0,0,Anc,20230904/MDS42_IS1.fa,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,L01_Anc,1,1,FACS,20231004/L01_Anc_m1.fasta,R01,1,L01-1,L01_Anc,20220418,True,20231004.0,20231004.0,NaN,L01_Anc_m1.fasta


In [4]:
is_pos_df = pd.read_csv(is_pos_path)
is_pos_df.head(2)

,start,end,IS_strand,max_alignment_length,length,cluster_id,Line,Gen
0,476928,480020,reverse,3093,3093,0,L01-1,1
1,557167,560259,reverse,3093,3093,1,L01-1,1


In [5]:
clip_df = pd.read_csv(clip_df_path)
clip_df.head(2)

,Bin,ClippingCount,ClippingCountRaw,ReadDepth,ClippingRate,Chromosome,End,Original,Cluster
0,0,30,30,30.137275,0.995445,L01-1_G08,1,0,1
1,4051000,30,30,29.480952,1.017606,L01-1_G08,1,0,2


In [6]:
ids.gen.unique()

array(['Anc', 'FACS', '8', '20'], dtype=object)

In [7]:
valid_ids = ids.query('gen in ["8", "20"]').IS_Detect_ID
clip_df_sub = clip_df.query('ClippingRate > 0.6 & End == 0 & Original == 0')
clip_df_sub = clip_df_sub.query('Chromosome in @valid_ids')
clip_df_sub.shape

(5, 9)

In [8]:
# add one example of clipping at the cutadapt region
clip_df_sub = pd.concat([clip_df.query('Chromosome == "L01-1_G20" & End == 0 & Original == 1'), clip_df_sub]).reset_index(drop=True)

In [9]:
clip_df_sub

,Bin,ClippingCount,ClippingCountRaw,ReadDepth,ClippingRate,Chromosome,End,Original,Cluster
0,1415000,7,11,4.145,1.688782,L01-1_G20,0,1,2
1,1945000,11,11,14.835,0.741490,L01-4_G20,0,0,3
2,2082000,10,10,12.998,0.769349,L01-4_G20,0,0,4
3,1508000,16,16,22.101,0.723949,L05-2_G20,0,0,2
4,1709000,4,4,5.691,0.702864,L05-4_G08,0,0,2
5,1189000,11,11,18.198,0.604462,L06-1_G08,0,0,2


In [10]:
unique_Clusters = clip_df_sub[['Chromosome', 'Cluster']].drop_duplicates().\
	itertuples(index=False, name=None)
min_igv_range_size = 20000

bat_file_dir = os.path.join(bat_dir, 'ShowCliprichReads.bat')
with open(bat_file_dir, "w") as bat_file:
	# Write the IGV setup commands
	bat_file.write(f'snapshotDirectory {igv_snapshot_dir}\n')
	bat_file.write("preference DEFAULT_FONT_SIZE 20\n")
	for Chromosome, Cluster in unique_Clusters:
		bins = clip_df_sub.query('Chromosome==@Chromosome & Cluster==@Cluster')
		vis_range_start = bins.Bin.min() - min_igv_range_size//2 -1
		vis_range_end = bins.Bin.max() + min_igv_range_size//2
		sample_ =Chromosome 

		clip_id = f'{Chromosome}_{Cluster}_{vis_range_start}-{vis_range_end}'

		# create bed file for IGV with strand
		line_ = Chromosome.split('_')[0]
		gen_ = Chromosome.split('_')[1]
		gen_ = 2 if gen_ == 'G08' else 3 if gen_ == 'G20' else 0
		bed_df = is_pos_df.query(f'Line == "{line_}" & Gen == {gen_}')
		bed_df['chr'] = sample_
		bed_df['strand'] = bed_df.IS_strand.apply(lambda x: '+' if x == 'forward' else '-')
		bed_df['IS_ID'] = bed_df.cluster_id.apply(lambda x: f'{line_}.{gen_}.{x}')
		bed_df['start'] = bed_df['start'] - 1
		bed_df = bed_df[['chr', 'start', 'end', 'IS_ID', 'length', 'strand']]
		bed_df.to_csv(os.path.join(bed_dir, f'{sample_}.bed'), sep='\t', header=False, index=False)

		# Generate IGV commands
		bat_file.write("new\n")
		bat_file.write(f"genome {igv_fasta_dir}\\{sample_}.fasta\n")
		bat_file.write(f"load {igv_bam_dir}\\{sample_}.bam\n")
		bat_file.write(f"load {igv_bam_dir2}\\{sample_}.bam\n")
		bat_file.write(f"load {igv_bed_dir}\\{sample_}.bed\n")
		bat_file.write(f"goto {sample_}:{vis_range_start}-{vis_range_end}\n")
		bat_file.write(f"collapse {sample_}.bam\n")
		bat_file.write(f"maxPanelHeight 600\n")
		bat_file.write(f"expand {sample_}.bam\n")
		bat_file.write(f"setColor 155,155,155 {sample_}.bam\n")
		#bat_file.write(f"setTrackHeight 1000 {sample_}.bam\n")
		bat_file.write(f"setTrackHeight 15 {sample_}.bed\n")
		bat_file.write(f"snapshot {clip_id}.png\n")
		bat_file.write(f"snapshot {clip_id}.svg\n")
		#bat_file.write(f"saveSession {clip_id}.xml\n")
		# fullpath
		bat_file.write(f"savesession {igv_snapshot_dir}\\{clip_id}.xml\n")

		bat_file.write(f"\n\n")

/tmp/ipykernel_19100/1785497222.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['chr'] = sample_
/tmp/ipykernel_19100/1785497222.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['strand'] = bed_df.IS_strand.apply(lambda x: '+' if x == 'forward' else '-')
/tmp/ipykernel_19100/1785497222.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://